# MarianMT Fine-Tuning: Local vs Google Colab Speed Comparison

This notebook compares the speed of fine-tuning a MarianMT English-Tagalog translation model on:
- **Local CPU** (your current setup)
- **Google Colab GPU** (T4/V100/A100)

## Speed Analysis Summary:
- **Local CPU**: ~20+ days for 1M samples
- **Colab T4 GPU**: ~5-10 hours for 1M samples (50-100x faster)
- **Colab Pro GPU**: ~2-5 hours for 1M samples (100-200x faster)

**Recommendation**: Use Google Colab for serious fine-tuning!

In [ ]:
# Section 1: Import Libraries and Set Up Logging
import os
import time
import logging
from datasets import load_dataset, Dataset
from transformers import MarianMTModel, MarianTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
import torch

# Configure logging for notebook output
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("marianmt_fine_tune")

print("📦 Libraries imported successfully!")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🚀 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎯 GPU device: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Section 2: Load Pretrained MarianMT Model and Tokenizer
model_name = "Helsinki-NLP/opus-mt-en-tl"

try:
    logger.info(f"Loading model: {model_name}")
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name)
    
    # Move model to GPU if available
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    
    logger.info(f"✅ Model loaded successfully on {device}")
    print(f"📊 Model parameters: {model.num_parameters():,}")
    
except Exception as e:
    logger.error(f"❌ Error loading model: {e}")
    raise

In [ ]:
# Section 3: Load and Validate Parallel Corpus

# Configuration options
USE_SAMPLE_DATA = False  # Set to False for full dataset
SAMPLE_SIZE = 10000     # Reduce for testing (use 10k-50k for quick tests)

# Load corpus - adjust path for your environment
# For local: use your corpus.csv path
# For Colab: upload corpus.csv to /content/ or mount Google Drive
if 'google.colab' in str(get_ipython()):
    # Running in Colab - upload corpus.csv or mount Drive
    corpus_path = "/content/corpus.csv"  # Upload corpus.csv to Colab
    print("🔄 Upload your corpus.csv file to /content/ directory")
else:
    # Running locally
    corpus_path = "baybayin_backend/data/translation/corpus.csv"

try:
    logger.info(f"Loading corpus from: {corpus_path}")
    dataset = load_dataset('csv', data_files={'train': corpus_path}, split='train')
    
    print(f"📊 Dataset loaded: {len(dataset):,} rows")
    print(f"📋 Columns: {dataset.column_names}")
    print(f"🔍 Sample data:")
    for i in range(min(3, len(dataset))):
        print(f"  EN: {dataset[i]['english'][:100]}...")
        print(f"  TL: {dataset[i]['tagalog'][:100]}...")
        print()
        
except Exception as e:
    logger.error(f"❌ Error loading corpus: {e}")
    print("💡 For Colab: Make sure to upload corpus.csv to /content/")
    raise

In [ ]:
# Section 4: Filter and Preprocess Dataset

def is_valid(row):
    """Check if a row has valid English and Tagalog text"""
    return (bool(row.get('english')) and 
            bool(row.get('tagalog')) and 
            isinstance(row['english'], str) and 
            isinstance(row['tagalog'], str) and
            len(row['english'].strip()) > 0 and
            len(row['tagalog'].strip()) > 0)

# Filter valid rows
valid_rows = [row for row in dataset if is_valid(row)]
logger.info(f"📊 Valid rows: {len(valid_rows):,} out of {len(dataset):,}")

# Sample data for faster training (optional)
if USE_SAMPLE_DATA and len(valid_rows) > SAMPLE_SIZE:
    import random
    valid_rows = random.sample(valid_rows, SAMPLE_SIZE)
    logger.info(f"🎯 Using sample: {len(valid_rows):,} rows for faster training")

if len(valid_rows) == 0:
    raise ValueError("❌ No valid training samples found. Check data format.")

# Convert to Dataset
dataset = Dataset.from_list(valid_rows)
print(f"✅ Preprocessed dataset ready: {len(dataset):,} samples")

In [ ]:
# Section 5: Tokenize Dataset

def preprocess(examples):
    """Tokenize English inputs and Tagalog targets"""
    inputs = [str(x) for x in examples['english'] if x is not None]
    targets = [str(x) for x in examples['tagalog'] if x is not None]
    
    # Skip empty batches
    if not inputs or not targets or len(inputs) != len(targets):
        return {}
    
    # Tokenize inputs and targets
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding=True)
    labels = tokenizer(targets, max_length=128, truncation=True, padding=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

logger.info("🔄 Tokenizing dataset...")
start_time = time.time()

tokenized_dataset = dataset.map(preprocess, batched=True, remove_columns=dataset.column_names)

tokenization_time = time.time() - start_time
logger.info(f"✅ Tokenization complete in {tokenization_time:.1f}s")
print(f"📊 Tokenized dataset: {len(tokenized_dataset):,} samples")

In [ ]:
# Section 6: Configure Training Arguments

# Output directory
output_dir = "/content/en-tl-marianmt" if 'google.colab' in str(get_ipython()) else "./en-tl-marianmt"
os.makedirs(output_dir, exist_ok=True)

# Training configuration optimized for speed
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,  # Reduced for faster training
    per_device_train_batch_size=16 if torch.cuda.is_available() else 4,  # Larger batch on GPU
    gradient_accumulation_steps=2,  # Effective batch size = batch_size * gradient_accumulation_steps
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir=f"{output_dir}/logs",
    logging_steps=100,
    save_strategy="epoch",
    eval_strategy="no",
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),  # Enable FP16 on GPU for faster training
    dataloader_pin_memory=False,  # Can help with data loading speed
    remove_unused_columns=True,
)

print(f"🎯 Training configuration:")
print(f"  Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  FP16: {training_args.fp16}")
print(f"  Output: {output_dir}")

# Calculate estimated training time
total_samples = len(tokenized_dataset)
effective_batch_size = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
total_steps = (total_samples // effective_batch_size) * training_args.num_train_epochs

if torch.cuda.is_available():
    estimated_time_hours = total_steps / 3600  # ~1 step/sec on GPU
    print(f"⏱️  Estimated training time: {estimated_time_hours:.1f} hours (on GPU)")
else:
    estimated_time_hours = total_steps * 5 / 3600  # ~5 sec/step on CPU
    print(f"⏱️  Estimated training time: {estimated_time_hours:.1f} hours (on CPU)")
    print(f"⚠️  Consider using Google Colab GPU for much faster training!")

In [ ]:
# Section 7: Initialize Trainer

# Data collator for sequence-to-sequence models
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

# Initialize trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,  # Updated parameter name
    data_collator=data_collator,
)

logger.info("✅ Trainer initialized successfully")
print(f"🎯 Ready to train on {len(tokenized_dataset):,} samples")

In [ ]:
# Section 8: Train Model and Save Outputs

try:
    logger.info("🚀 Starting training...")
    print("=" * 50)
    print("🔥 TRAINING STARTED")
    print("=" * 50)
    
    # Start training
    training_start = time.time()
    trainer.train()
    training_end = time.time()
    
    training_duration = training_end - training_start
    
    print("=" * 50)
    print("✅ TRAINING COMPLETED")
    print("=" * 50)
    
    logger.info(f"🎉 Training completed in {training_duration:.1f} seconds ({training_duration/60:.1f} minutes)")
    
    # Save model and tokenizer
    logger.info("💾 Saving model and tokenizer...")
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    logger.info(f"✅ Model saved to: {output_dir}")
    
    # Test the model with a sample translation
    print("\n🧪 Testing the fine-tuned model:")
    test_text = "Hello, how are you?"
    inputs = tokenizer(test_text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=128, num_beams=4)
    translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"EN: {test_text}")
    print(f"TL: {translation}")
    
except Exception as e:
    logger.error(f"❌ Training error: {e}")
    raise

In [ ]:
# Section 9: Measure Training Time

def format_time(seconds):
    """Format seconds into human-readable time"""
    if seconds < 60:
        return f"{seconds:.1f} seconds"
    elif seconds < 3600:
        return f"{seconds/60:.1f} minutes"
    else:
        return f"{seconds/3600:.1f} hours"

if 'training_duration' in locals():
    print("⏱️  TRAINING PERFORMANCE SUMMARY")
    print("=" * 40)
    print(f"📊 Dataset size: {len(tokenized_dataset):,} samples")
    print(f"⏱️  Total training time: {format_time(training_duration)}")
    print(f"🚀 Speed: {len(tokenized_dataset)/training_duration:.1f} samples/second")
    print(f"🎯 Device used: {'GPU' if torch.cuda.is_available() else 'CPU'}")
    
    if torch.cuda.is_available():
        print(f"🔥 GPU: {torch.cuda.get_device_name(0)}")
        print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
    # Estimate time for full dataset
    full_dataset_size = 1_072_536  # Your original dataset size
    if len(tokenized_dataset) < full_dataset_size:
        estimated_full_time = (training_duration / len(tokenized_dataset)) * full_dataset_size
        print(f"\n📈 Estimated time for full dataset ({full_dataset_size:,} samples):")
        print(f"   {format_time(estimated_full_time)}")
else:
    print("❌ Training not completed yet. Run the training cell first.")

# Section 10: Compare Training Speed: Local vs. Colab

## 🏃‍♂️ Speed Comparison Results

Based on your current local training and this notebook's results:

### Your Local CPU Performance:
- **Hardware**: Local CPU
- **Speed**: 185 steps in 22 minutes = ~8.2 steps/minute
- **Estimated time for full dataset**: 20+ days
- **Recommendation**: ❌ Too slow for production

### Google Colab Performance:
Run this notebook in Google Colab to see GPU performance:

1. **Google Colab (Free T4 GPU)**:
   - Expected speed: 50-100x faster than CPU
   - Estimated time: 5-10 hours for full dataset
   - Cost: Free (with usage limits)

2. **Google Colab Pro (V100/A100 GPU)**:
   - Expected speed: 100-200x faster than CPU  
   - Estimated time: 2-5 hours for full dataset
   - Cost: $10/month

## 📋 How to run in Google Colab:

1. Go to [colab.research.google.com](https://colab.research.google.com)
2. Upload this notebook
3. Upload your `corpus.csv` to `/content/`
4. Change runtime to GPU: Runtime → Change runtime type → GPU
5. Run all cells

## 💡 Optimization Tips:

- **Start small**: Use 10k-50k samples for testing
- **Use GPU**: Always enable GPU runtime in Colab
- **Batch size**: Increase batch size on GPU (16-32)
- **Mixed precision**: Enable FP16 for faster training
- **Reduce epochs**: Start with 1 epoch, then increase if needed